# C2P-CLIP Training on AIGC Detection Benchmark

This notebook trains (or fine-tunes) the **C2P-CLIP** detector on the `TheKernel01/AIGC-Detection-Benchmark` HuggingFace dataset, which contains:

- **Real** images (label `0`)
- **AI-generated** images from 17 generators (label `1`–`17`)

The dataset has `train`, `validation`, and `test` splits.

### Architecture summary
- Backbone: `openai/clip-vit-large-patch14` (frozen)
- Head: a single linear layer `fc: 768 → 1` trained with BCE loss
- Task: binary classification — **Real vs. AI-generated**

## 1. Environment Setup

In [1]:
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn import metrics
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import CLIPModel
from tqdm.auto import tqdm

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

PyTorch version : 2.3.0+cu121
CUDA available  : True
GPU             : NVIDIA GeForce GTX 1650 Ti


## 2. Configuration

In [2]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 123


def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Dataset ───────────────────────────────────────────────────────────────────
HF_DATASET = 'TheKernel01/AIGC-Detection-Benchmark'
HF_TOKEN = None  # set to your token string, or read from env below
CACHE_DIR = None  # e.g. '/path/to/hf_cache'

# Read token from .env / environment variable if not set above
if HF_TOKEN is None:
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass
    HF_TOKEN = os.getenv('HF_TOKEN')

# ── Training hyperparameters ──────────────────────────────────────────────────
CLIP_MODEL_NAME = 'openai/clip-vit-large-patch14'
IMAGE_SIZE = 224
BATCH_SIZE = 64  # reduce to 32 if OOM
NUM_EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4

# ── Checkpointing ────────────────────────────────────────────────────────────
CHECKPOINT_DIR = './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── CLIP normalization (NOT ImageNet defaults) ────────────────────────────────
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

# ── Generator id → name mapping ──────────────────────────────────────────────
GENERATOR_MAPPING = {
    0: 'Real',
    1: 'ADM',
    2: 'BigGAN',
    3: 'CycleGAN',
    4: 'DALLE2',
    5: 'GauGAN',
    6: 'Glide',
    7: 'Midjourney',
    8: 'ProGAN',
    9: 'SD14',
    10: 'SD15',
    11: 'SDXL',
    12: 'StarGAN',
    13: 'StyleGAN',
    14: 'StyleGAN2',
    15: 'VQDM',
    16: 'WhichFaceIsReal',
    17: 'Wukong',
}

print('Configuration set.')

Configuration set.


## 3. Data Loading

In [3]:
print('Loading dataset splits from HuggingFace...')
train_hf = load_dataset(HF_DATASET, split='train', token=HF_TOKEN, cache_dir=CACHE_DIR)
val_hf = load_dataset(
    HF_DATASET, split='validation', token=HF_TOKEN, cache_dir=CACHE_DIR
)
test_hf = load_dataset(HF_DATASET, split='test', token=HF_TOKEN, cache_dir=CACHE_DIR)

print(f'Train      : {len(train_hf):,} samples')
print(f'Validation : {len(val_hf):,} samples')
print(f'Test       : {len(test_hf):,} samples')

Loading dataset splits from HuggingFace...


Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Train      : 75,014 samples
Validation : 12,498 samples
Test       : 37,514 samples


In [4]:
class AIGCDataset(Dataset):
    """Wraps a HuggingFace AIGC split.

    Returns (image_tensor, binary_label) where binary_label=0 for real,
    and binary_label=1 for any AI-generated image.
    """

    def __init__(self, hf_data, transform=None):
        self.hf_data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.hf_data)

    def __getitem__(self, idx):
        item = self.hf_data[idx]
        image = item['image'].convert('RGB')
        # Collapse multi-class generator labels → binary: 0=real, 1=fake
        binary_label = 0.0 if item['label'] == 0 else 1.0
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(binary_label, dtype=torch.float32)

In [5]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ]
)

# ── Datasets & DataLoaders ────────────────────────────────────────────────────
train_dataset = AIGCDataset(train_hf, transform=train_transform)
val_dataset = AIGCDataset(val_hf, transform=eval_transform)
test_dataset = AIGCDataset(test_hf, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

Train batches : 1173
Val   batches : 196
Test  batches : 587


## 4. Model Definition

In [6]:
class C2P_CLIP(nn.Module):
    """CLIP ViT-L/14 backbone + single linear detection head.

    The vision encoder and projection are frozen; only `fc` is trained.
    """

    def __init__(self, name: str = CLIP_MODEL_NAME, num_classes: int = 1):
        super().__init__()
        self.model = CLIPModel.from_pretrained(name)

        # Drop text branch — not needed
        del self.model.text_model
        del self.model.text_projection
        del self.model.logit_scale

        # Freeze vision backbone
        self.model.vision_model.requires_grad_(False)
        self.model.visual_projection.requires_grad_(False)

        # Trainable detection head
        self.model.fc = nn.Linear(768, num_classes)
        nn.init.normal_(self.model.fc.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.model.fc.bias)

    def encode_image(self, img: torch.Tensor) -> torch.Tensor:
        vision_outputs = self.model.vision_model(
            pixel_values=img,
            output_attentions=self.model.config.output_attentions,
            output_hidden_states=self.model.config.output_hidden_states,
            return_dict=self.model.config.use_return_dict,
        )
        pooled = vision_outputs[1]
        return self.model.visual_projection(pooled)

    def forward(self, img: torch.Tensor) -> torch.Tensor:
        embeds = self.encode_image(img)
        embeds = embeds / embeds.norm(p=2, dim=-1, keepdim=True)  # L2-normalise
        return self.model.fc(embeds)  # raw logits, shape [B, 1]


model = C2P_CLIP(name=CLIP_MODEL_NAME, num_classes=1).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,} / {total:,}  ({100 * trainable / total:.1f}%)')

Trainable params : 769 / 303,966,977  (0.0%)


## 5. Loss, Optimizer & Scheduler

In [7]:
criterion = nn.BCEWithLogitsLoss()

# Only optimise the detection head
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# Cosine annealing over all training steps
total_steps = NUM_EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=LR * 0.01
)

print(f'Total training steps : {total_steps:,}')

Total training steps : 5,865


## 6. Evaluation Helpers

In [8]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    """Return loss, accuracy, and AP on the given DataLoader."""
    model.eval()
    total_loss, n = 0.0, 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images).squeeze(1)  # [B]
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(images)
        n += len(images)

        probs = logits.sigmoid().cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    preds = (all_probs >= 0.5).astype(int)
    acc = (preds == all_labels.astype(int)).mean()
    ap = metrics.average_precision_score(all_labels, all_probs)
    auc = metrics.roc_auc_score(all_labels, all_probs)
    avg_loss = total_loss / n

    return {'loss': avg_loss, 'acc': acc, 'ap': ap, 'auc': auc}


def format_metrics(d: dict) -> str:
    return (
        f'loss={d["loss"]:.4f}  acc={d["acc"] * 100:.1f}%  '
        f'AP={d["ap"] * 100:.1f}  AUC={d["auc"] * 100:.1f}'
    )

## 7. (Optional) Load Pre-trained Weights

Skip this cell to train from a randomly-initialised head, or uncomment to fine-tune from the released C2P-CLIP checkpoint.

In [9]:
# ── Option A: load from URL ────────────────────────────────────────────────────
# state_dict = torch.hub.load_state_dict_from_url(
#     'https://www.now61.com/f/95OefW/C2P_CLIP_release_20240901.zip',
#     map_location='cpu', progress=True,
# )
# model.load_state_dict(state_dict, strict=True)
# print('Loaded released C2P-CLIP weights.')

# ── Option B: load a local checkpoint ─────────────────────────────────────────
# ckpt_path = './checkpoints/best_model.pth'
# ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
# model.load_state_dict(ckpt['model_state_dict'])
# print(f'Loaded checkpoint from {ckpt_path}')

print('Skipping pre-trained weight loading — training from scratch.')

Skipping pre-trained weight loading — training from scratch.


## 8. Training Loop

In [10]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_ap': [], 'val_auc': []}
best_val_ap = 0.0
best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

print(f'Starting training for {NUM_EPOCHS} epoch(s) on {DEVICE}\n')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=True)
    for images, labels in pbar:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images).squeeze(1)  # [B]
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        pbar.set_postfix(
            loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}'
        )

    avg_train_loss = epoch_loss / len(train_loader)

    # ── Validation ───────────────────────────────────────────────────────────
    val_metrics = evaluate(model, val_loader)
    elapsed = time.time() - t0

    print(
        f'[Epoch {epoch:02d}]  train_loss={avg_train_loss:.4f}  '
        f'val: {format_metrics(val_metrics)}  ({elapsed:.0f}s)'
    )

    # ── Record history ────────────────────────────────────────────────────────
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['acc'])
    history['val_ap'].append(val_metrics['ap'])
    history['val_auc'].append(val_metrics['auc'])

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_metrics['ap'] > best_val_ap:
        best_val_ap = val_metrics['ap']
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_ap': val_metrics['ap'],
                'val_auc': val_metrics['auc'],
                'val_acc': val_metrics['acc'],
            },
            best_ckpt,
        )
        print(
            f'  ✓ New best AP={best_val_ap * 100:.2f} — checkpoint saved to {best_ckpt}'
        )

    # ── Epoch checkpoint ──────────────────────────────────────────────────────
    epoch_ckpt = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch:02d}.pth')
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, epoch_ckpt)

print('\nTraining complete.')

Starting training for 5 epoch(s) on cuda



Epoch 1/5:   0%|          | 0/1173 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'], label='Val')
axes[0].set_title('BCE Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, [a * 100 for a in history['val_acc']], color='green')
axes[1].set_title('Val Accuracy (%)')
axes[1].set_xlabel('Epoch')

axes[2].plot(
    epochs_range, [a * 100 for a in history['val_ap']], label='AP', color='orange'
)
axes[2].plot(
    epochs_range, [a * 100 for a in history['val_auc']], label='AUC', color='purple'
)
axes[2].set_title('Val AP & AUC (%)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=120)
plt.show()

## 10. Final Evaluation on Test Set

Load the best checkpoint and evaluate **overall** and **per-generator**.

In [ ]:
# ── Load best weights ─────────────────────────────────────────────────────────
ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
print(
    f'Loaded best checkpoint (epoch {ckpt["epoch"]}, val AP={ckpt["val_ap"] * 100:.2f}%)'
)

# ── Overall test metrics ──────────────────────────────────────────────────────
test_metrics = evaluate(model, test_loader)
print(f'\nTest  : {format_metrics(test_metrics)}')

In [ ]:
# ── Per-generator breakdown ──────────────────────────────────────────────────
print('\nPer-generator test results:')
print(f'{"Generator":<20}  {"N":>5}  {"Acc":>6}  {"AP":>6}  {"AUC":>6}')
print('-' * 50)

all_generators = np.array(test_hf['generator'])

model.eval()
with torch.no_grad():
    for gen_id, gen_name in GENERATOR_MAPPING.items():
        indices = np.nonzero(all_generators == gen_id)[0]
        if len(indices) == 0:
            continue

        # Build a subset loader for this generator only
        from torch.utils.data import Subset

        subset = Subset(test_dataset, indices)
        sub_loader = DataLoader(
            subset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )

        probs_list, labels_list = [], []
        for imgs, lbls in sub_loader:
            imgs = imgs.to(DEVICE)
            p = model(imgs).sigmoid().squeeze(1).cpu().numpy()
            probs_list.append(p)
            labels_list.append(lbls.numpy())

        probs = np.concatenate(probs_list)
        labels = np.concatenate(labels_list)
        preds = (probs >= 0.5).astype(int)

        acc = (preds == labels.astype(int)).mean() * 100

        # AP/AUC require both classes present in the subset
        if len(np.unique(labels)) > 1:
            ap = metrics.average_precision_score(labels, probs) * 100
            auc = metrics.roc_auc_score(labels, probs) * 100
        else:
            ap = auc = float('nan')

        label_str = f'{gen_name} (real)' if gen_id == 0 else f'{gen_name} (fake)'
        print(
            f'{label_str:<20}  {len(indices):>5}  {acc:>5.1f}%  {ap:>5.1f}  {auc:>5.1f}'
        )

## 11. Save Final Model for Inference

In [ ]:
final_path = os.path.join(CHECKPOINT_DIR, 'c2p_clip_final.pth')
torch.save(model.state_dict(), final_path)
print(f'Final model state dict saved to: {final_path}')

# ── Quick inference demo ──────────────────────────────────────────────────────
print('\nInference demo (first batch of test set):')
model.eval()
imgs, lbls = next(iter(test_loader))
with torch.no_grad():
    logits = model(imgs.to(DEVICE)).squeeze(1)
    probs = logits.sigmoid().cpu()
    preds = (probs >= 0.5).long()

for i in range(min(8, len(preds))):
    gt_str = 'Real' if lbls[i].item() == 0 else 'Fake'
    pred_str = 'Real' if preds[i].item() == 0 else 'Fake'
    print(
        f'  Sample {i + 1:02d}: GT={gt_str:<4}  Pred={pred_str:<4}  p(fake)={probs[i]:.3f}'
    )